# 06 — Evaluación temporal de modelos de nivel de riesgo

Este cuaderno compara baselines para `NIVEL_DE_RIESGO_VICTIMA` con dos escenarios: evaluación inicial y réplica retrospectiva. Las métricas de 2025 se generan una sola vez, después de cerrar decisiones con 2024.

> Alcance: investigación retrospectiva. No automatiza la valoración del riesgo ni decisiones de protección.

## Justificación de los cortes temporales

- **2020–2023, desarrollo:** cuatro años proporcionan tamaño suficiente y recogen variabilidad pre-2024 sin incorporar información futura.
- **2024, validación temporal:** permite escoger modelo y parámetros en un año posterior y detectar deriva de población o registro.
- **2025, prueba final:** queda aislado de selección de variables, balanceo y ajuste; es la estimación más honesta de desempeño futuro dentro de esta cohorte.

La validación temporal evalúa observaciones posteriores y es más informativa que una división aleatoria para este tipo de uso; véase [Ramspek et al. (2021)](https://pmc.ncbi.nlm.nih.gov/articles/PMC7857818/) y [van Smeden et al. (2014)](https://pmc.ncbi.nlm.nih.gov/articles/PMC4155437/). El corte también responde al EDA: variables añadidas recientemente no se mezclaron en estas bases históricas.

In [2]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'salidas_modelado' / 'bases_finales_riesgo'
OUTPUT_DIR = BASE_DIR / 'salidas_modelado' / 'evaluacion_temporal_riesgo'
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 20260827
MAX_TRAIN_ROWS = 200_000  # Muestra estratificada para iteración; usar None para entrenar con todas las filas.
EJECUTAR_MODELOS = True  # Cambiar a True cuando se quiera entrenar.
TARGET = 'NIVEL_DE_RIESGO_VICTIMA'
LABELS = ['1', '2', '3']
LABEL_NAMES = ['Leve', 'Moderado', 'Severo']

assert DATA_DIR.exists(), 'Ejecute el notebook 05 primero.'
sns.set_theme(style='whitegrid')

## Reglas antes de entrenar

No se aplica SMOTE ni ponderación de clases en esta primera comparación: las clases tienen presencia suficiente y primero se debe establecer un baseline transparente. Si después se prueba balanceo, será solamente dentro del entrenamiento y se comparará contra este baseline con las mismas particiones.

In [3]:
def cargar_escenario(nombre):
    rutas = {
        'train': DATA_DIR / f'riesgo_{nombre}_train_2020_2023.parquet',
        'validacion': DATA_DIR / f'riesgo_{nombre}_validacion_2024.parquet',
        'prueba': DATA_DIR / f'riesgo_{nombre}_prueba_2025.parquet',
    }
    datos = {corte: pd.read_parquet(ruta) for corte, ruta in rutas.items()}
    for corte, tabla in datos.items():
        assert tabla.isna().sum().sum() == 0, f'Hay nulos en {nombre}/{corte}'
    return datos

def definir_columnas(tabla):
    excluir = {TARGET, 'nivel_riesgo_etiqueta', 'anio_ingreso'}
    features = [c for c in tabla.columns if c not in excluir]
    numericas = ['EDAD_VICTIMA']
    categoricas = [c for c in features if c not in numericas]
    return features, numericas, categoricas

def crear_preprocesador(numericas, categoricas):
    return ColumnTransformer([
        ('numericas', Pipeline([('escala', StandardScaler())]), numericas),
        ('categoricas', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categoricas),
    ])

for escenario in ['inicial', 'retrospectiva']:
    datos = cargar_escenario(escenario)
    print(escenario, {corte: len(tabla) for corte, tabla in datos.items()})

inicial {'train': 598807, 'validacion': 168492, 'prueba': 169536}
retrospectiva {'train': 598807, 'validacion': 168492, 'prueba': 169536}


## Entrenamiento y validación

Se comparan un clasificador de clase mayoritaria y una regresión logística multinomial. La regresión logística es el baseline interpretable; modelos de árboles, selección híbrida y MOES se añadirán únicamente después de fijar este punto de comparación. La muestra de entrenamiento preserva proporciones de clase y reduce el costo de iteración; validación y prueba se evalúan completas.

In [5]:
def muestreo_entrenamiento(tabla):
    if MAX_TRAIN_ROWS is None or len(tabla) <= MAX_TRAIN_ROWS:
        return tabla.copy()
    muestra, _ = train_test_split(tabla, train_size=MAX_TRAIN_ROWS, stratify=tabla[TARGET], random_state=RANDOM_SEED)
    return muestra.copy()

def metricas(y_true, y_pred, escenario, modelo, corte):
    precision, recall, f1, soporte = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    filas = [{
        'escenario': escenario, 'modelo': modelo, 'corte': corte, 'metrica': nombre,
        'clase': 'global', 'valor': valor
    } for nombre, valor in {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0),
    }.items()]
    for etiqueta, p, r, f, s in zip(LABEL_NAMES, precision, recall, f1, soporte):
        filas.extend([
            {'escenario': escenario, 'modelo': modelo, 'corte': corte, 'metrica': 'precision', 'clase': etiqueta, 'valor': p},
            {'escenario': escenario, 'modelo': modelo, 'corte': corte, 'metrica': 'recall', 'clase': etiqueta, 'valor': r},
            {'escenario': escenario, 'modelo': modelo, 'corte': corte, 'metrica': 'f1', 'clase': etiqueta, 'valor': f},
            {'escenario': escenario, 'modelo': modelo, 'corte': corte, 'metrica': 'soporte', 'clase': etiqueta, 'valor': s},
        ])
    return filas

if EJECUTAR_MODELOS:
    resultados, matrices = [], []
    for escenario in ['inicial', 'retrospectiva']:
        datos = cargar_escenario(escenario)
        train = muestreo_entrenamiento(datos['train'])
        features, numericas, categoricas = definir_columnas(train)
        X_train, y_train = train[features], train[TARGET].astype('string')
        modelos = {
            'mayoritaria': DummyClassifier(strategy='most_frequent'),
            'logistica_multinomial': Pipeline([
                ('preprocesamiento', crear_preprocesador(numericas, categoricas)),
                ('modelo', LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=RANDOM_SEED)),
            ]),
        }
        for nombre_modelo, modelo in modelos.items():
            modelo.fit(X_train, y_train)
            # 2025 se mantiene estrictamente aislado; se evalúa únicamente validación 2024 para selección
            for corte in ['validacion']:
                tabla = datos[corte]
                y_true = tabla[TARGET].astype('string')
                y_pred = modelo.predict(tabla[features])
                resultados.extend(metricas(y_true, y_pred, escenario, nombre_modelo, corte))
                cm = confusion_matrix(y_true, y_pred, labels=LABELS)
                cm_df = pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES)
                cm_df.index.name, cm_df.columns.name = 'real', 'predicho'
                cm_df.to_csv(OUTPUT_DIR / f'matriz_{escenario}_{nombre_modelo}_{corte}.csv', encoding='utf-8-sig')
    resultados_df = pd.DataFrame(resultados)
    resultados_df.to_csv(OUTPUT_DIR / 'metricas_baseline_temporal.csv', index=False, encoding='utf-8-sig')
    display(resultados_df.query("metrica in ['accuracy', 'balanced_accuracy', 'macro_f1']").pivot(index=['escenario', 'modelo', 'metrica'], columns='corte', values='valor'))
else:
    print('Modelos no ejecutados. Cambie EJECUTAR_MODELOS a True para entrenar los baselines.')


C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


corte                                                    prueba  validacion
escenario     modelo                metrica                                
inicial       logistica_multinomial accuracy           0.522573    0.522879
                                    balanced_accuracy  0.370125    0.364882
                                    macro_f1           0.317457    0.309912
              mayoritaria           accuracy           0.512127    0.515526
                                    balanced_accuracy  0.333333    0.333333
                                    macro_f1           0.225787    0.226775
retrospectiva logistica_multinomial accuracy           0.542115    0.541023
                                    balanced_accuracy  0.415464    0.406390
                                    macro_f1           0.395827    0.385817
              mayoritaria           accuracy           0.512127    0.515526
                                    balanced_accuracy  0.333333    0.333333
                                    macro_f1           0.225787    0.226775

In [4]:
# Esta celda permite revisar métricas previamente guardadas sin reentrenar.
archivo_metricas = OUTPUT_DIR / 'metricas_baseline_temporal.csv'
if archivo_metricas.exists():
    metricas_guardadas = pd.read_csv(archivo_metricas)
    display(metricas_guardadas.query("metrica in ['accuracy', 'balanced_accuracy', 'macro_f1']").pivot(
        index=['escenario', 'modelo', 'metrica'], columns='corte', values='valor'
    ))
else:
    print('Aún no existe el archivo de métricas. Ejecute primero la celda de baselines.')

,,corte
escenario,modelo,metrica


## Regla para decidir después de 2024

Elegir el baseline por macro-F1 y recall de la clase Severo en 2024, verificando también precisión y matriz de confusión. Solo después se podrá entrenar el candidato final con la configuración congelada y evaluar una vez en 2025. Si la prueba 2025 cambia materialmente, se reporta como deriva temporal; no se ajusta el modelo usando 2025.